# Watch brush selections arrive in Python
Run the cell below, then **drag horizontally across the chart**. The current range and matching row IDs update while you drag. **Release the mouse** to save a snapshot in the Python list `regions`; the history below updates automatically.

`latest_selection` holds the current snapshot and `selected_rows` holds the example rows inside its x range. The API returns a range; this example filters the rows in Python. Values are kept in notebook memory, not written to disk. Rerunning the cell starts a fresh history.

In [ ]:
import asyncio
import genome_spy as gs
from IPython.display import Markdown, display

# Cancel pending setup and close the previous display when rerunning.
if previous_task := globals().get("connection_task"):
    previous_task.cancel()
if previous_widget := globals().get("widget"):
    previous_widget.close()

rows = [
    {"id": name, "x": x, "y": y}
    for x, (name, y) in enumerate(
        zip("ABCDEFGHIJ", [2, 5, 3, 7, 4, 6, 2, 8, 5, 3]), start=1
    )
]
brush = gs.selection_interval("brush", encodings=["x"])
chart = (
    gs.Chart(rows)
    .mark_point(size=100)
    .encode(x="x:Q", y="y:Q")
    .properties(width=600, height=220)
)
chart = chart.add_params(brush)
widget = chart.widget(inline=True, controls=False)
display(widget)

live_output = display(Markdown("Connecting to the chart…"), display_id=True)
history_output = display(Markdown("**Saved selections: 0**"), display_id=True)
regions = []
latest_selection = None
selection = None
selected_rows = []


def show_current(snapshot):
    global latest_selection, selected_rows
    latest_selection = snapshot
    interval = snapshot["intervals"].get("x")
    selected_rows = (
        [row for row in rows if interval[0] <= row["x"] <= interval[1]]
        if interval is not None
        else []
    )
    current_range = (
        f"{interval[0]:.2f} – {interval[1]:.2f}"
        if interval is not None
        else "No active selection — drag across the chart."
    )
    ids = ", ".join(row["id"] for row in selected_rows) or "none"
    live_output.update(
        Markdown(
            f"**Current x range:** {current_range}\n\n"
            f"**Matching row IDs (filtered in Python):** {ids}"
        )
    )


def save_selection(snapshot):
    regions.append(snapshot)
    entries = []
    for number, saved in enumerate(regions[-5:], start=max(1, len(regions) - 4)):
        interval = saved["intervals"].get("x")
        label = f"x = {interval[0]:.2f} – {interval[1]:.2f}" if interval else "cleared"
        entries.append(f"{number}. {label}")
    history_output.update(
        Markdown(
            f"**Saved selections: {len(regions)}** — latest five snapshots in `regions`\n\n"
            + "\n".join(entries)
        )
    )


async def connect():
    global api, selection, stop_live, stop
    try:
        async with asyncio.timeout(30):
            api = await widget.get_embed_api()
            selection = await api.params.get_selection("brush")
            stop_live = await selection.subscribe(show_current)
            stop = await selection.subscribe(save_selection, {"delivery": "commit"})
            show_current(await selection.get_value())
    except Exception as error:
        live_output.update(
            Markdown(
                f"**Connection failed:** {type(error).__name__}: {error}. Selections are not being saved."
            )
        )


# Let the cell finish so VS Code/Jupyter can process browser replies.
connection_task = asyncio.create_task(connect())

### Inspect or control the same selection
After brushing, run the next cell to inspect the values saved in Python. Rerun it after another brush to refresh this snapshot; the displays above update automatically. These values stay in memory, not on disk.

Once connected, clear the brush **from Python** with `clear_task = asyncio.create_task(selection.clear())`. The live display and history respond too. Use background tasks for browser calls in VS Code; directly awaiting a reply in a running cell can block the messages needed to complete it. To stop listening, schedule `stop_live()` and `stop()` the same way.


In [ ]:
display(
    {
        "current_selection": latest_selection,
        "selected_rows": selected_rows,
        "saved_count": len(regions),
        "saved_selections": regions,
    }
)

### Save a named brush annotation
This is a separate example with its own chart and brush, directly above the form. Run the cell, brush **this chart**, enter a name and optional description, then click **Save annotation**. Saved ranges appear here and in the table; the first chart is unaffected. `annotations` holds the records in Python memory, not on disk. Rerunning this cell resets them. Adapted from the [GenomeSpy selection form example](https://genomespy.app/docs/api/embed-examples/selectionForm/).


In [ ]:
import asyncio
import html
import genome_spy as gs
import ipywidgets as widgets
from IPython.display import display

for task_name in ("annotation_connection_task", "annotation_save_task"):
    if previous_task := globals().get(task_name):
        previous_task.cancel()
if previous_widget := globals().get("annotation_widget"):
    previous_widget.close()

annotation_points = (
    gs.Chart(
        [
            {"x": x, "y": y}
            for x, y in enumerate([2, 5, 3, 7, 4, 6, 2, 8, 5, 3], start=1)
        ]
    )
    .mark_point(size=100)
    .encode(x="x:Q", y="y:Q")
    .properties(width=600, height=220)
)
annotation_track = (
    gs.Chart(data={"name": "annotations"})
    .mark_rect(fill="#0ea5e9", fillOpacity=0.25)
    .encode(x="start:Q", x2="end:Q")
)
annotation_chart = gs.vconcat(
    annotation_points, annotate=[annotation_track], datasets={"annotations": []}
).add_params(gs.selection_interval("brush", encodings=["x"]))
annotations = []
annotation_selection = None
annotation_widget = annotation_chart.widget(inline=True, controls=False)
display(annotation_widget)

annotation_name = widgets.Text(description="Name:")
annotation_description = widgets.Textarea(description="Description:")
save_button = widgets.Button(description="Save annotation", disabled=True)
annotation_status = widgets.HTML("Connecting to the annotation chart…")
annotation_table = widgets.HTML()


def render_annotations():
    annotation_table.value = (
        "<table><tr><th>Range</th><th>Name</th><th>Description</th></tr>"
        + "".join(
            f"<tr><td>{item['start']:.2f}–{item['end']:.2f}</td>"
            f"<td>{html.escape(item['name'])}</td>"
            f"<td>{html.escape(item['description'])}</td></tr>"
            for item in annotations
        )
        + "</table>"
    )


async def save_annotation():
    save_button.disabled = True
    try:
        if (
            not annotation_connection_task.done()
            or annotation_connection_task.cancelled()
            or annotation_selection is None
        ):
            raise ValueError("Wait for the chart to connect first.")
        name = annotation_name.value.strip()
        if not name:
            raise ValueError("Enter an annotation name.")
        async with asyncio.timeout(30):
            snapshot = await annotation_selection.get_value()
            interval = snapshot["intervals"].get("x")
            if not interval:
                raise ValueError("Brush a region first.")
            record = {
                "start": interval[0],
                "end": interval[1],
                "name": name,
                "description": annotation_description.value.strip(),
            }
            await annotation_api.datasets.set("annotations", [*annotations, record])
            annotations.append(record)
            render_annotations()
            annotation_status.value = (
                f"Saved {len(annotations)} annotation(s) in Python memory."
            )
            annotation_name.value = ""
            annotation_description.value = ""
            await annotation_selection.clear()
    except Exception as error:
        annotation_status.value = html.escape(f"{type(error).__name__}: {error}")
    finally:
        save_button.disabled = False


def start_save():
    global annotation_save_task
    previous_save = globals().get("annotation_save_task")
    if previous_save is not None and not previous_save.done():
        return
    annotation_save_task = asyncio.create_task(save_annotation())


def on_save_clicked(button):
    # JupyterLab may deliver widget clicks on a separate event loop.
    annotation_connection_task.get_loop().call_soon_threadsafe(start_save)


save_button.on_click(on_save_clicked)
render_annotations()
display(
    widgets.VBox(
        [
            annotation_name,
            annotation_description,
            save_button,
            annotation_status,
            annotation_table,
        ]
    )
)


async def connect_annotations():
    global annotation_api, annotation_selection
    try:
        async with asyncio.timeout(30):
            annotation_api = await annotation_widget.get_embed_api()
            annotation_selection = await annotation_api.params.get_selection("brush")
        annotation_status.value = "Ready — brush the chart directly above this form."
        save_button.disabled = False
    except Exception as error:
        annotation_status.value = html.escape(
            f"Connection failed: {type(error).__name__}: {error}"
        )


annotation_connection_task = asyncio.create_task(connect_annotations())